In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install lz4 mtcnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 82.8 MB/s eta 0:00:00


In [ ]:
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 19.3 MB/s eta 0:00:00


In [ ]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 86.7 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


New code

In [ ]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
from mtcnn import MTCNN
from google.colab import drive
from collections import defaultdict

# 2. Initialize MTCNN
detector = MTCNN()

# --- Configuration ---
# 1. The 50-subject folder created previously (to get the list of names)
downsampled_dir = '/content/drive/MyDrive/UNIS6/cv/ds_1'

# 2. The full AgeDB folder (to search for all possible images of those 50 subjects)
full_dataset_dir = '/content/drive/MyDrive/UNIS6/cv/AgeDB'

# 3. Final Destination
drive_output_dir = '/content/drive/MyDrive/UNIS6/cv/ID_1'
os.makedirs(drive_output_dir, exist_ok=True)

def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_metrics_mtcnn(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    h_img, w_img = img.shape[:2]

    if h_img < 80 or w_img < 80: return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        results = detector.detect_faces(img_rgb)
    except Exception:
        return None

    if not results: return None

    valid_faces = [r for r in results if r['box'][2] > 50 and r['box'][3] > 50]
    if not valid_faces: return None

    best_face = max(valid_faces, key=lambda x: x['confidence'])
    keypoints = best_face['keypoints']
    x, y, w, h = best_face['box']

    x, y = max(0, x), max(0, y)
    face_img = img[y:y+h, x:x+w]
    if face_img.size == 0: return None

    # Pose, Sharpness, and Size
    nose = keypoints['nose']
    l_eye = keypoints['left_eye']
    r_eye = keypoints['right_eye']
    pose_score = abs(euclidean(nose, l_eye) - euclidean(nose, r_eye))

    gray_face = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    sharpness_score = cv2.Laplacian(gray_face, cv2.CV_64F).var()
    size_score = (w * h) / (h_img * w_img)

    return {"pose": pose_score, "sharpness": sharpness_score, "size": size_score}

# --- Main Execution ---
if os.path.exists(downsampled_dir) and os.path.exists(full_dataset_dir):

    # 1. Identify the 50 Target Subjects from the downsampled folder
    print("Identifying target subjects from Downsampled folder...")
    downsampled_files = [f for f in os.listdir(downsampled_dir) if f.lower().endswith(('.jpeg', '.jpg'))]

    target_subjects = set()
    for f in downsampled_files:
        parts = f.split('_')
        if len(parts) >= 2:
            target_subjects.add(parts[1])

    print(f"Targeting {len(target_subjects)} subjects.")

    # 2. Group ALL files from full dataset by subject, but ONLY for target subjects
    subject_groups = defaultdict(list)
    all_files = [f for f in os.listdir(full_dataset_dir) if f.lower().endswith(('.jpeg', '.jpg'))]

    print(f"Scanning full dataset for images of target subjects...")
    for f in all_files:
        parts = f.split('_')
        if len(parts) >= 2:
            subject_name = parts[1]
            if subject_name in target_subjects:
                subject_groups[subject_name].append(f)

    print(f"Found matches. Starting MTCNN selection across {len(subject_groups)} subjects...")
    found_count = 0

    # 3. Iterate through each unique subject group and find the best photo
    for subject_name, files in tqdm(subject_groups.items()):
        best_file_name = None
        best_full_path = None
        best_score = -float('inf')

        for f in files:
            path = os.path.join(full_dataset_dir, f)
            m = calculate_metrics_mtcnn(path)

            if m is None: continue

            # Weighted Scoring (Same as your previous logic)
            # High Size + High Sharpness - High Pose (Asymmetry)
            current_score = (m['size'] * 100) - (m['pose'] * 10) + (m['sharpness'] * 0.05)

            if current_score > best_score:
                best_score = current_score
                best_full_path = path
                best_file_name = f

        if best_full_path:
            dest_path = os.path.join(drive_output_dir, best_file_name)
            shutil.copy(best_full_path, dest_path)
            found_count += 1

    print(f"\n✅ Done! {found_count} best frontal images saved to {drive_output_dir}")
else:
    print("Error: Ensure both the Downsampled directory and Full Dataset directory paths are correct.")

Identifying target subjects from Downsampled folder...
Targeting 50 subjects.
Scanning full dataset for images of target subjects...
Found matches. Starting MTCNN selection across 50 subjects...


100%|██████████| 50/50 [03:35<00:00,  4.32s/it]


✅ Done! 50 best frontal images saved to /content/drive/MyDrive/UNIS6/cv/ID_1


local_data_dir = '/content/drive/MyDrive/UNIS6/cv/Downsamples'
drive_output_dir = '/content/drive/MyDrive/UNIS6/cv/ID_imgs'